# 03 - Negative Reduction Audit

This notebook audits the reduced datasets used for modelling. It does not train models. It checks split sizes, color-space representativeness, and holdout completeness.


## Operational Reading

- `train`: training WR rows and reduced training negatives.
- `threshold_calibration`: remaining non-holdout negatives used for threshold stress-testing.
- `holdout`: final test split created before negative reduction; it preserves the natural imbalance.
- SMOTE/SMOTE-ENN is not applied here. It is applied later inside the modelling pipeline.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Force an interactive notebook backend when running cells manually.
try:
    get_ipython().run_line_magic('matplotlib', 'inline')
except NameError:
    pass
from IPython.display import Markdown, Image, display

try:
    import seaborn as sns
except ImportError:
    sns = None

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.22,
    "font.size": 10,
})

PALETTE = {
    "xgboost": "#2F6F9F",
    "random_forest": "#3A9D5D",
    "hist_gradient_boosting": "#B7791F",
    "logistic_regression": "#7B61A8",
    "wr": "#A8324A",
    "candidate": "#2F6F9F",
    "negative": "#6B7280",
    "muted": "#6B7280",
}

def require_path(path):
    path = Path(path)
    if not path.is_absolute():
        path = ROOT / path
    if not path.exists():
        raise FileNotFoundError(path)
    return path

def pct(value):
    return "-" if pd.isna(value) else f"{100 * float(value):.1f}%"

def model_display(row):
    variant = str(row.get("dataset_variant", ""))
    family = "Relaxed" if variant.startswith("relaxed") else "Strict"
    dataset = variant.replace("relaxed_", "").replace("strict_", "")
    dataset = dataset.replace("photometry", "Phot").replace("parallax_soft", "Parallax+").replace("poe_", "POE>=")
    feat = "+err" if str(row.get("feature_set", "")).endswith("error") else "base"
    model = {"xgboost": "XGB", "random_forest": "RF", "hist_gradient_boosting": "HGB", "logistic_regression": "LogReg"}.get(str(row.get("model", "")), str(row.get("model", "")))
    sampler = {"smote": "SMOTE", "smote_enn": "SMOTE-ENN", "none": "no sampler"}.get(str(row.get("sampler", "")), str(row.get("sampler", "")))
    return f"{model} {sampler} | {family} {dataset} | {feat}"

def add_model_columns(df):
    out = df.copy()
    out["model_label"] = out.apply(model_display, axis=1)
    for k in [50, 100, 500, 1000]:
        c = f"holdout_wr_at_{k}"
        if c in out.columns:
            out[f"{c}_pct"] = out[c] / out["wr_holdout"].replace(0, np.nan)
    return out

def rank_models(df, k=100):
    return df.sort_values(
        [f"holdout_wr_at_{k}_pct", "holdout_average_precision", "holdout_recall_at_100", "holdout_f2_wr"],
        ascending=[False, False, False, False],
    ).reset_index(drop=True)

from wr_detector.config import load_yaml

config = load_yaml(ROOT / "configs/models.yaml")
modeling_dir = require_path(config["outputs"]["modeling_data_dir"])
summary = pd.read_csv(require_path(config["outputs"]["reduction_summary"]))
audit = pd.read_csv(require_path(config["outputs"]["reduction_audit"]))
pool_path = ROOT / config["outputs"].get("negative_pool_audit", "reports/tables/negative_pool_audit.csv")
pool_audit = pd.read_csv(pool_path) if pool_path.exists() else pd.DataFrame()


## Split Dimensions By Variant


In [ ]:
rows = []
for path in sorted(modeling_dir.glob("*_reduced.parquet")):
    variant = path.stem.replace("_reduced", "")
    df = pd.read_parquet(path, columns=["target", "modeling_split"])
    for split, group in df.groupby("modeling_split"):
        rows.append({
            "dataset_variant": variant,
            "split": split,
            "rows": len(group),
            "wr": int(group["target"].sum()),
            "non_wr": int((group["target"] == 0).sum()),
        })
dims = pd.DataFrame(rows)
display(dims.pivot_table(index="dataset_variant", columns="split", values=["rows", "wr", "non_wr"], aggfunc="sum").fillna(0).astype(int))

plot = dims.pivot(index="dataset_variant", columns="split", values="rows").fillna(0)
plot = plot[[c for c in ["train", "threshold_calibration", "holdout"] if c in plot.columns]]
fig, ax = plt.subplots(figsize=(12, 5.6))
colors = ["#3A9D5D", "#9CA3AF", "#2F6F9F"]
bottom = np.zeros(len(plot))
for col, color in zip(plot.columns, colors):
    label = "calibration" if col == "threshold_calibration" else col
    ax.bar(plot.index, plot[col], bottom=bottom, color=color, label=label)
    bottom += plot[col].to_numpy()
ax.set_title("Rows by split after negative reduction")
ax.set_ylabel("rows")
ax.tick_params(axis="x", rotation=35)
ax.legend(frameon=False)
plt.tight_layout()
plt.show()


## Train-Only Negative Reduction


In [ ]:
view = summary.copy()
view["train_neg_per_wr"] = view["train_negative_reduced"] / view["train_wr"].replace(0, np.nan)
view["holdout_wr_pct"] = view["holdout_wr"] / view["holdout_rows"].replace(0, np.nan)
display(view[[
    "dataset_variant", "train_wr", "train_negative_original", "train_negative_reduced",
    "train_neg_per_wr", "threshold_calibration_negative", "holdout_rows", "holdout_wr",
    "holdout_negative", "holdout_wr_pct"
]].style.format({"train_neg_per_wr": "{:.2f}", "holdout_wr_pct": "{:.3%}"}))

fig, ax = plt.subplots(figsize=(11, 5))
plot = view.sort_values("train_neg_per_wr")
ax.barh(plot["dataset_variant"], plot["train_neg_per_wr"], color="#3A9D5D")
ax.axvline(10, color="#111827", ls="--", lw=1.2, label="10x target")
for y, val in enumerate(plot["train_neg_per_wr"]):
    ax.text(val + 0.12, y, f"{val:.1f}x", va="center", fontsize=9)
ax.set_xlabel("reduced negatives / train WR")
ax.set_title("Negative reduction is applied only to training rows")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()


## Color-Space Representativeness


In [ ]:
display(audit.sort_values("ks_statistic", ascending=False).head(24).style.format(precision=4))

fig, ax = plt.subplots(figsize=(11.5, 5.5))
plot = audit.sort_values("ks_statistic", ascending=False).head(24).iloc[::-1]
ax.barh(plot["dataset_variant"] + " | " + plot["color"], plot["ks_statistic"], color="#2F6F9F")
ax.set_title("Largest KS differences after negative reduction")
ax.set_xlabel("KS statistic")
plt.tight_layout()
plt.show()


## Negative Pool Audit


In [ ]:
if pool_audit.empty:
    display(Markdown("Missing `negative_pool_audit.csv`."))
else:
    heat = pool_audit.pivot_table(index="dataset_variant", columns="pool", values="ks_vs_all_negative", aggfunc="max")
    display(pool_audit.sort_values("ks_vs_all_negative", ascending=False).head(25).style.format(precision=4))
    fig, ax = plt.subplots(figsize=(8.5, 5))
    if sns:
        sns.heatmap(heat, annot=True, fmt=".3f", cmap="Blues", ax=ax, cbar_kws={"label": "max KS"})
    else:
        im = ax.imshow(heat.fillna(0), cmap="Blues")
        ax.set_xticks(range(len(heat.columns)), heat.columns, rotation=30, ha="right")
        ax.set_yticks(range(len(heat.index)), heat.index)
        plt.colorbar(im, ax=ax, label="max KS")
    ax.set_title("Maximum difference by negative pool")
    plt.tight_layout()
    plt.show()


## Histograms Before And After Reduction


In [ ]:
VARIANT = "strict_photometry"
COLORS = ["BP_RP", "J_K", "W1_W2"]
df = pd.read_parquet(modeling_dir / f"{VARIANT}_reduced.parquet")
neg = df[df["target"].eq(0)]
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=False)
for ax, color in zip(axes, COLORS):
    parts = {
        "reduced train": neg[neg["modeling_split"].eq("train")][color].dropna(),
        "calibration": neg[neg["modeling_split"].eq("threshold_calibration")][color].dropna(),
        "holdout": neg[neg["modeling_split"].eq("holdout")][color].dropna(),
    }
    bins = np.histogram_bin_edges(pd.concat(list(parts.values())), bins=40)
    ax.hist(parts["calibration"], bins=bins, density=True, alpha=0.35, color="#9CA3AF", label="calibration")
    ax.hist(parts["holdout"], bins=bins, density=True, alpha=0.35, color="#2F6F9F", label="holdout")
    ax.hist(parts["reduced train"], bins=bins, density=True, histtype="step", lw=2, color="#3A9D5D", label="reduced train")
    ax.set_title(color)
    ax.set_xlabel(color)
axes[0].set_ylabel("density")
axes[-1].legend(frameon=False)
fig.suptitle(f"Negative distributions by pool - {VARIANT}", y=1.03)
plt.tight_layout()
plt.show()


## Color-color checks


In [ ]:
VARIANT = "strict_photometry"
df = pd.read_parquet(modeling_dir / f"{VARIANT}_reduced.parquet")
wr = df[df["target"].eq(1)]
neg = df[df["target"].eq(0)]
if len(neg) > 3500:
    neg = neg.sample(3500, random_state=42)
planes = [("G_BP", "G_RP"), ("J_H", "J_K"), ("BP_RP", "W1_W2")]
fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
for ax, (x, y) in zip(axes, planes):
    for split, color, alpha in [("threshold_calibration", "#9CA3AF", 0.22), ("holdout", "#2F6F9F", 0.28), ("train", "#3A9D5D", 0.55)]:
        part = neg[neg["modeling_split"].eq(split)]
        ax.scatter(part[x], part[y], s=8, alpha=alpha, color=color, label=split if (x, y) == planes[0] else None)
    ax.scatter(wr[x], wr[y], s=26, color=PALETTE["wr"], edgecolor="white", linewidth=0.4, label="WR" if (x, y) == planes[0] else None)
    ax.set_xlabel(x)
    ax.set_ylabel(y)
    ax.set_title(f"{x} vs {y}")
axes[0].legend(frameon=False)
fig.suptitle(f"Color-color coverage after reduction - {VARIANT}", y=1.03)
plt.tight_layout()
plt.show()
